In [2]:
import os
import sys
import django

# Add project root to path and configure Django
sys.path.insert(0, os.path.dirname(os.getcwd()))
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'settings.settings')
os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = 'true'
django.setup()

In [3]:
import pandas as pd
from jobs.models import Job, ExtendedJob

## 1. Load Raw Data

In [4]:
# Query jobs that have extended job data using select_related for efficiency
jobs_with_extended = ExtendedJob.objects.select_related('job').values(
    'job__title',
    'job_category',
    'job__company',
    'job__location',
    'job__salary',
    'bachelor_required',
    'master_required',
    'phd_required',
    'tech_stack',
    'min_experience_years',
    'us_only',
    'employment_type',
    'medical_insurance',
)

# Convert to DataFrame
df = pd.DataFrame(list(jobs_with_extended))

# Rename columns to remove the 'job__' prefix
df.columns = [
    'title',
    'job_category',
    'company',
    'location',
    'salary',
    'bachelor_required',
    'master_required',
    'phd_required',
    'tech_stack',
    'min_experience_years',
    'us_only',
    'employment_type',
    'medical_insurance',
]

df

,title,job_category,company,location,salary,bachelor_required,master_required,phd_required,tech_stack,min_experience_years,us_only,employment_type,medical_insurance
0,Service Desk Support Analyst III - Kelsey - Se...,HD,Optum,", US, TXPearland",$28.94 - $51.63 an hour,True,False,False,"windows,active directory,microsoft office,wan,...",4,True,full-time,True
1,Principal Enterprise Architect (Remote),SSWE,IQVIA,", US, NJWayne","$118,100 - $328,800 a year",True,False,False,"Kafka,Apigee,Azure API Management,Kong,Azure M...",15,True,full-time,True
2,Senior Technical Program Manager (ST PM) (15.35),PM,"OCT Consulting, LLC",", US, DCWashington","$175,000 - $250,000 a year",True,False,False,"aws,cloud,cybersecurity,agile,itil,cmmi,pmboK",8,True,full-time,True
3,Cloud Cybersecurity Manager (CCM) (15.35),DevOps,"OCT Consulting, LLC",", US, DCWashington","$150,000 - $225,000 a year",True,False,False,"aws,cloud,cybersecurity,zerotrust,nist,stigs,s...",8,True,full-time,True
4,Cloud Architect and Infrastructure Lead (CAIL)...,DevOps,"OCT Consulting, LLC",", US, DCWashington","$150,000 - $200,000 a year",True,False,False,"aws,devsecops,agile,cybersecurity",6,True,full-time,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1907,"Senior Staff Software Engineer, Backend (Consu...",SSWE,Coinbase,", USRemote","$246,500 - $290,000 a year",True,False,False,"go,temporal,kubernetes,mongodb",10,False,full-time,True
1908,"Staff Software Engineer, Backend (Consumer - T...",SSWE,Coinbase,", USRemote","$218,025 - $256,500 a year",False,False,False,"golang,clickhouse,kafka,redis,mongodb",8,False,full-time,True
1909,Engineering Manager (Consumer - Growth),EM,Coinbase,", USRemote","$218,025 - $256,500 a year",False,False,False,NaN,7,False,full-time,True
1910,"Senior Staff Software Engineer, Backend (Consu...",SSWE,Coinbase,", USRemote","$253,895 - $298,700 a year",False,False,False,NaN,0,False,full-time,True


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1912 entries, 0 to 1911
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   title                 1912 non-null   str   
 1   job_category          1912 non-null   str   
 2   company               1912 non-null   str   
 3   location              1912 non-null   str   
 4   salary                1912 non-null   str   
 5   bachelor_required     1912 non-null   bool  
 6   master_required       1912 non-null   bool  
 7   phd_required          1912 non-null   bool  
 8   tech_stack            1637 non-null   str   
 9   min_experience_years  1912 non-null   int64 
 10  us_only               1912 non-null   bool  
 11  employment_type       1845 non-null   str   
 12  medical_insurance     1907 non-null   object
dtypes: bool(4), int64(1), object(1), str(7)
memory usage: 142.0+ KB


## 2. Clean Cols

### 2.1 Clean: Salary

In [6]:
print(df['salary'].sample(30).to_string())

448     $160,000 - $195,000 a year
579     $126,500 - $140,000 a year
845       $57,784 - $85,000 a year
1012    $130,000 - $160,000 a year
421     $130,800 - $190,000 a year
87      $180,000 - $275,000 a year
711       $55,000 - $65,000 a year
1892    $139,027 - $145,000 a year
780       $55,000 - $62,000 a year
1003    $107,757 - $181,563 a year
1286    $175,000 - $185,000 a year
1729     $86,300 - $165,800 a year
980      $82,940 - $182,549 a year
218      $86,800 - $124,000 a year
532       $65,000 - $80,000 a year
1512     $90,000 - $129,500 a year
268      $85,093 - $103,500 a year
159                $120,000 a year
278     $110,000 - $140,000 a year
1377    $156,000 - $209,000 a year
565     $110,000 - $150,000 a year
275     $171,900 - $300,800 a year
667      $80,000 - $103,600 a year
1079             $70 - $80 an hour
1443    $112,000 - $141,000 a year
1671    $129,000 - $179,000 a year
794       $75,000 - $83,000 a year
1058    $200,000 - $240,000 a year
270     $110,000 - $

In [7]:
import re

def clean_salary_to_hourly(salary_str):
    """
    Convert salary string to hourly rate (float).
    - Removes $ and , symbols
    - Averages MIN-MAX ranges
    - Converts monthly (200-40k) and annual (>40k) to hourly
    """
    if pd.isna(salary_str) or not salary_str:
        return None
    
    # Remove $ and , symbols
    cleaned = salary_str.replace('$', '').replace(',', '')
    
    # Extract all numbers (including decimals)
    numbers = re.findall(r'[\d.]+', cleaned)
    
    if not numbers:
        return None
    
    # Convert to floats
    values = [float(n) for n in numbers]
    
    # Average if range (MIN-MAX)
    salary = sum(values) / len(values)
    
    # Convert to hourly based on value thresholds
    if salary < 200:
        # Already hourly
        hourly = salary
    elif salary < 40000:
        # Monthly -> divide by 173.33
        hourly = salary / 173.33
    else:
        # Annual -> divide by 2080
        hourly = salary / 2080
    
    return round(hourly, 2)

# Apply the cleaning function
df['salary_hourly'] = df['salary'].apply(clean_salary_to_hourly)

# Show sample of original vs cleaned
df[['salary', 'salary_hourly']].head(10)

,salary,salary_hourly
0,$28.94 - $51.63 an hour,40.29
1,"$118,100 - $328,800 a year",107.43
2,"$175,000 - $250,000 a year",102.16
3,"$150,000 - $225,000 a year",90.14
4,"$150,000 - $200,000 a year",84.13
5,"$120,001 - $160,000 a year",67.31
6,"$140,000 - $165,000 a year",73.32
7,"$111,000 - $156,500 a year",64.30
8,"$105,000 - $125,000 a year",55.29
9,"$45,000 - $110,000 a year",37.26


In [8]:
df["salary"] = df["salary_hourly"]
df.drop(columns="salary_hourly", inplace=True)

### 2.2 Clean: Employment type

In [9]:
df['employment_type'].unique()

<StringArray>
[         'full-time',           'contract',           'flexible',
                  nan,         'internship',          'part-time',
 'full-time,contract']
Length: 7, dtype: str

In [10]:
# Clean employment_type: keep only the first value if multiple exist
df['employment_type'] = df['employment_type'].str.split(',').str[0]

df['employment_type'].unique()

array(['full-time', 'contract', 'flexible', nan, 'internship',
       'part-time'], dtype=object)

### 2.3 Clean: Location

In [11]:
df['location'].unique()

<StringArray>
[     ', US, TXPearland',         ', US, NJWayne',    ', US, DCWashington',
       ', US, TXEl Paso',        ', US, TXIrving',        ', US, TXAustin',
  ', US, CAPort Hueneme',  ', US, COFort Collins',       ', US, VAFairfax',
       ', US, WAPullman',
 ...
       ', US, MIZeeland',          ', US, MSnull',     ', US, FLLake Mary',
    ', US, MASomerville',    ', US, OHCincinnati', ', US, OKOklahoma City',
   ', US, MNMaple Grove', ', US, ILDowners Grove',     ', US, NYGreenwich',
    ', US, ALBirmingham']
Length: 296, dtype: str

In [12]:
print(df['location'].sample(30).to_string())

1130              , US, INnull
8         , US, CAPort Hueneme
332                 , USRemote
276     , US, NCRaleigh-Durham
180                   , USnull
1694                , USRemote
522          , US, CASan Diego
1397     , US, CASan Francisco
1064            , US, VAMcLean
526      , US, CASan Francisco
898                 , USRemote
1372                , USRemote
1859            , US, VAMcLean
651                 , USRemote
303             , US, VATysons
809                   , USnull
292           , US, WABellevue
456                 , USRemote
1494                , USRemote
563                 , USRemote
660                   , USnull
716             , US, MANewton
1841                , USRemote
1512           , US, NJTeaneck
1005                , USRemote
1410            , US, VAMcLean
1248            , US, AZTucson
551                 , USRemote
1823              , US, VTnull
1363                , USRemote


In [13]:
def clean_location(loc_str):
    """
    Clean location to format: "US" or "US, Remote" or "US, STATE"
    
    Input formats:
    - ', US, TXPearland' -> 'US, TX'
    - ', USRemote' -> 'US, Remote'
    - ', US, MSnull' -> 'US, MS'
    - ', USnull' -> 'US'
    """
    if pd.isna(loc_str) or not loc_str:
        return 'US'
    
    # Remove leading comma and spaces
    cleaned = loc_str.strip().lstrip(',').strip()
    
    # Check for Remote
    if 'Remote' in cleaned:
        return 'US, Remote'
    
    # Check for just "USnull" (no state)
    if cleaned == 'USnull':
        return 'US'
    
    # Pattern: "US, STATEcity" - extract state code (2 letters after "US, ")
    if ', ' in cleaned:
        parts = cleaned.split(', ')
        if len(parts) >= 2:
            state_city = parts[1]
            # Extract first 2 characters as state code
            state = state_city[:2].upper()
            return f'US, {state}'
    
    return 'US'

# Apply the cleaning function
df['location'] = df['location'].apply(clean_location)

# Check unique values
df['location'].unique()

<StringArray>
[    'US, TX',     'US, NJ',     'US, DC',     'US, CA',     'US, CO',
     'US, VA',     'US, WA',     'US, MN',     'US, UT',     'US, FL',
     'US, GA',     'US, MA',     'US, IL',     'US, OH',     'US, WV',
     'US, AZ',     'US, SC',     'US, NC', 'US, Remote',         'US',
     'US, IN',     'US, MD',     'US, MI',     'US, NY',     'US, KY',
     'US, ME',     'US, NV',     'US, NM',     'US, PA',     'US, OR',
     'US, NH',     'US, CT',     'US, MO',     'US, WI',     'US, OK',
     'US, TN',     'US, KS',     'US, LA',     'US, RI',     'US, NE',
     'US, ND',     'US, IA',     'US, MT',     'US, VT',     'US, ID',
     'US, AL',     'US, WY',     'US, MS']
Length: 48, dtype: str

### 2.4 Clean: Medical Insurance

In [14]:
# Fill None values with False and convert to bool
df['medical_insurance'] = df['medical_insurance'].fillna(False).astype(bool)

df['medical_insurance'].dtype

dtype('bool')

## 3. EDA

### 3.1 NA/None counts for all columns

In [15]:
df.isna().sum()

title                     0
job_category              0
company                   0
location                  0
salary                    0
bachelor_required         0
master_required           0
phd_required              0
tech_stack              275
min_experience_years      0
us_only                   0
employment_type          67
medical_insurance         0
dtype: int64

### 3.2 AVG salary 

In [16]:
print(df['salary'].mean())

71.75748953974896


In [17]:
df.groupby('job_category')['salary'].mean().sort_values(ascending=False).round(2)

job_category
C         105.40
EM         97.45
SSWE       89.40
MLE        87.42
UI         79.79
DE         73.11
PM         71.22
TL         71.02
SWE        70.82
DevOps     67.91
QA         59.31
Other      53.71
DA         51.89
HD         39.60
Name: salary, dtype: float64

### 3.3 job_category count

In [18]:
df['job_category'].value_counts()

job_category
SWE       329
PM        310
DevOps    239
Other     174
SSWE      171
MLE       130
DA        121
DE        117
EM         95
HD         67
C          46
UI         44
QA         35
TL         34
Name: count, dtype: int64

### 3.4 Education level count

In [39]:
education_counts = pd.DataFrame({
    'bachelor_required': [df['bachelor_required'].sum()],
    'master_required': [df['master_required'].sum()],
    'phd_required': [df['phd_required'].sum()]
}).T
education_counts.columns = ['count']
education_counts['percentage'] = (education_counts['count'] / len(df) * 100).round(2)
education_counts

,count,percentage
bachelor_required,1079,56.43
master_required,50,2.62
phd_required,2,0.10


### 3.5 Education level average salary

In [40]:
education_salary = pd.DataFrame({
    'bachelor_required': [df[df['bachelor_required']]['salary'].mean()],
    'master_required': [df[df['master_required']]['salary'].mean()],
    'phd_required': [df[df['phd_required']]['salary'].mean()]
}).T
education_salary.columns = ['avg_salary']
education_salary.round(2)

,avg_salary
bachelor_required,73.26
master_required,90.75
phd_required,47.64


### 3.6 Company count

In [41]:
df['company'].value_counts().head(20)

company
CVS Health                                 27
Humana                                     23
World Wide Technology Holding, LLC         21
General Dynamics Information Technology    20
Gainwell Technologies LLC                  19
Concentrix                                 18
Atlassian                                  17
Netflix                                    15
Centene                                    14
Dropbox                                    14
Optum                                      13
FBG Enterprises Opco, LLC                  10
Waymo                                      10
MetroStar                                  10
NTT DATA                                   10
SAIC                                        9
GitHub                                      9
Odixcity Consulting                         9
The College Board                           9
ATPCO                                       9
Name: count, dtype: int64

### 3.7 Company average salary

In [42]:
df.groupby('company')['salary'].mean().sort_values(ascending=False).head(20).round(2)

company
Vi-M Professional Solutions           360.58
Odixcity Consulting                   240.38
Bigged                                240.38
PyStrap Technologies                  240.38
Fave Technologies Limited             240.38
Netflix                               228.04
Advanced Monitored Caregiving Inc.    210.00
59 Pines                              168.27
SupplyHouse                           164.90
Koloxo West Africa                    156.25
YIP Online Limited                    156.25
MCG Health                            154.90
Prolific                              150.00
Veeam Software                        149.18
GoPro                                 146.73
HighLevel                             145.43
Dr. Berg Nutritionals                 145.00
BHG Financial                         144.23
RSM                                   143.63
Alteryx                               137.02
Name: salary, dtype: float64

## 4. Encoding

In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1912 entries, 0 to 1911
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   title                 1912 non-null   str    
 1   job_category          1912 non-null   str    
 2   company               1912 non-null   str    
 3   location              1912 non-null   str    
 4   salary                1912 non-null   float64
 5   bachelor_required     1912 non-null   bool   
 6   master_required       1912 non-null   bool   
 7   phd_required          1912 non-null   bool   
 8   tech_stack            1637 non-null   str    
 9   min_experience_years  1912 non-null   int64  
 10  us_only               1912 non-null   bool   
 11  employment_type       1845 non-null   object 
 12  medical_insurance     1912 non-null   bool   
dtypes: bool(5), float64(1), int64(1), object(1), str(5)
memory usage: 129.0+ KB


In [34]:
from sklearn.preprocessing import MultiLabelBinarizer
from category_encoders import TargetEncoder

# Create a copy for encoding
df_encoded = df.copy()

# 1. Drop title (too noisy, job_category already captures role info)
df_encoded = df_encoded.drop(columns=['title'])

# 2. Target encode high/medium cardinality categoricals
target_cols = ['job_category', 'company', 'location']
te = TargetEncoder(cols=target_cols, smoothing=10)
df_encoded[target_cols] = te.fit_transform(df_encoded[target_cols], df_encoded['salary'])

# 3. One-hot encode employment_type (low cardinality, handles NaN)
df_encoded = pd.get_dummies(df_encoded, columns=['employment_type'], dummy_na=True)

# 4. Multi-label binarize tech_stack
df_encoded['tech_stack'] = df_encoded['tech_stack'].fillna('')
tech_lists = df_encoded['tech_stack'].str.lower().str.split(',')

# Filter to top N most common technologies to reduce noise
from collections import Counter
all_techs = [tech.strip() for techs in tech_lists for tech in techs if tech.strip()]
tech_counts = Counter(all_techs)
top_techs = {tech for tech, count in tech_counts.most_common(50)}  # Keep top 50

# Filter tech lists to only include top techs
tech_lists_filtered = tech_lists.apply(lambda x: [t.strip() for t in x if t.strip() in top_techs])

mlb = MultiLabelBinarizer()
tech_encoded = pd.DataFrame(
    mlb.fit_transform(tech_lists_filtered),
    columns=[f'tech_{t}' for t in mlb.classes_],
    index=df_encoded.index
)
df_encoded = pd.concat([df_encoded.drop(columns=['tech_stack']), tech_encoded], axis=1)

# 5. Convert ALL bool columns to int (explicit for XGBoost)
bool_cols = df_encoded.select_dtypes(include='bool').columns.tolist()
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

df_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 1912 entries, 0 to 1911
Data columns (total 66 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   job_category                1912 non-null   float64
 1   company                     1912 non-null   float64
 2   location                    1912 non-null   float64
 3   salary                      1912 non-null   float64
 4   bachelor_required           1912 non-null   int64  
 5   master_required             1912 non-null   int64  
 6   phd_required                1912 non-null   int64  
 7   min_experience_years        1912 non-null   int64  
 8   us_only                     1912 non-null   int64  
 9   medical_insurance           1912 non-null   int64  
 10  employment_type_contract    1912 non-null   int64  
 11  employment_type_flexible    1912 non-null   int64  
 12  employment_type_full-time   1912 non-null   int64  
 13  employment_type_internship  1912 non-null   

In [35]:
# Preview encoded features
df_encoded.head()

,job_category,company,location,salary,bachelor_required,master_required,phd_required,min_experience_years,us_only,medical_insurance,...,tech_react,tech_rest,tech_salesforce,tech_snowflake,tech_sql,tech_sql server,tech_tableau,tech_terraform,tech_typescript,tech_windows
0,39.885112,76.603802,64.223308,40.29,1,0,0,4,1,1,...,0,0,0,0,0,0,0,0,0,1
1,89.403562,75.517340,68.551588,107.43,1,0,0,15,1,1,...,0,0,0,1,0,0,0,0,0,0
2,71.221387,75.314720,70.115357,102.16,1,0,0,8,1,1,...,0,0,0,0,0,0,0,0,0,0
3,67.910042,75.314720,70.115357,90.14,1,0,0,8,1,1,...,0,0,0,0,0,0,0,0,0,0
4,67.910042,75.314720,70.115357,84.13,1,0,0,6,1,1,...,0,0,0,0,0,0,0,0,0,0


## 5. Correlation Matrix

In [37]:
# Calculate correlation with salary (target variable)
correlations = df_encoded.corr()['salary'].drop('salary').sort_values(key=abs, ascending=False)

# Display top correlations
print("Top 20 features correlated with salary (by absolute value):\n")
print(correlations.head(20).round(2).to_string())
print("\n" + "="*50)
print("\nBottom 10 (weakest correlations):\n")
print(correlations.tail(10).round(2).to_string())

Top 20 features correlated with salary (by absolute value):

company                       0.84
job_category                  0.42
min_experience_years          0.34
location                      0.20
employment_type_nan           0.15
tech_excel                   -0.15
tech_python                   0.13
tech_windows                 -0.12
employment_type_internship   -0.11
tech_gcp                      0.11
tech_go                       0.11
tech_jira                    -0.10
master_required               0.09
tech_c++                      0.09
tech_ai                       0.09
tech_aws                      0.09
tech_kafka                    0.09
tech_html                    -0.08
tech_java                     0.08
tech_cloud                    0.08


Bottom 10 (weakest correlations):

tech_agile                  0.02
tech_api                   -0.02
tech_javascript            -0.02
tech_jenkins                0.01
tech_rest                  -0.01
employment_type_contract    0.01
tech